# Loading the data:

In [1]:
# load the cleaned dataset
from ReusableFunc import load_clean_data


PrData = load_clean_data(r"C:\Users\Hi\Desktop\Risk-Analysis\data\processed\PrData.csv")
print("Data loaded successfully!")

Data loaded successfully!


# Statistical Analysis

# 1. What is the typical patients Age, Length of stay and Treatment cost?

In [2]:
PrData[['Age', 'Length_of_Stay', 'Treatment_Cost']].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,495.0,54.119192,20.992478,18.0,37.0,54.0,73.0,90.0
Length_of_Stay,505.0,4.677228,4.241328,1.0,2.0,4.0,6.0,60.0
Treatment_Cost,505.0,85868.843564,32249.055342,17712.0,62720.0,81476.0,104372.0,193348.0


Descriptive analysis was conducted to understand the distribution of key numerical variables. The average patient age was 54.1 years (SD = 21.0), with ages ranging from 18 to 90 years. The median age was 54 years. 

Hospital length of stay averaged 4.68 days (SD = 4.24), with a median of 4 days and a range of 1 to 60 days. The maximum length of stay of 60 days appears substantially higher than the typical stay and should be investigated as a potential outlier. 

Treatment costs averaged approximately KSh 85,869, with a median of KSh 81,476 and a standard deviation of KSh 32,249, indicating considerable variation in treatment expenditure.

# 2. Is Readmission associated with Diabetes, gender, Admission type and discharge type?

In [28]:
import pandas as pd

# Create a crosstab to analyze the relationship between Diabetes and Readmission within 30 days
pd.crosstab(
    PrData['Diabetes'],
    PrData['Readmitted_30_Days'],
    normalize='index'
).mul(100).round(2)

# Create a crosstab to analyze the relationship between Gender and Readmission within 30 days
#pd.crosstab(
   # PrData['Gender'],
    #PrData['Readmitted_30_Days'],
   # normalize='index'
#).mul(100).round(2) 

Readmitted_30_Days,No,Yes
Diabetes,,
No,79.67,20.33
Yes,66.18,33.82


Diabetes vs 30-Day Readmission

The cross-tabulation indicates a difference in 30-day readmission between patients with and without diabetes. Among patients without diabetes, 20.33% were readmitted within 30 days, while 79.67% were not readmitted. In contrast, among patients with diabetes, 33.82% were readmitted and 66.18% were not readmitted. This suggests that patients with diabetes had a higher proportion of 30-day readmissions compared with patients without diabetes.

Key finding
Diabetes	Not readmitted	Readmitted
No	79.67%	20.33%
Yes	66.18%	33.82%

So, readmission was about 13.49 percentage points higher among patients with diabetes (33.82% vs 20.33%).

However, this is only a descriptive finding. To determine whether the relationship between diabetes and readmission is statistically significant, you should follow it with a chi-square test of independence.

# 3. Chi-square test

The chi-square test is appropriate when you want to determine whether two categorical variables are statistically associated.

For example:

Diabetes vs 30-day readmission

In [29]:
from scipy.stats import chi2_contingency

table = pd.crosstab(
    PrData['Diabetes'],
    PrData['Readmitted_30_Days']
)

chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("p-value:", p)

Chi-square: 9.21095172462051
p-value: 0.0024057159503890193


A chi-square test of independence was conducted to assess the association between diabetes status and 30-day hospital readmission. The test showed a statistically significant association between diabetes and 30-day readmission (χ² = 9.211, p = 0.002). Patients with diabetes had a higher readmission rate (33.82%) compared with patients without diabetes (20.33%). Therefore, diabetes status appears to be associated with 30-day readmission in this dataset.

In conclusion: Diabetes was significantly associated with 30-day readmission

# 5. Correlation Analysis:

Does length of stay increase as treatment cost increase?

In [30]:
PrData[['Length_of_Stay', 'Treatment_Cost']].corr()

,Length_of_Stay,Treatment_Cost
Length_of_Stay,1.000000,0.641052
Treatment_Cost,0.641052,1.000000


There was a moderately strong positive correlation between Length of Stay and Treatment Cost (r = 0.641). This indicates that patients with longer hospital stays tended to incur higher treatment costs. The finding suggests that length of stay may be an important factor associated with treatment expenditure. However, correlation does not imply causation, and other patient and treatment-related factors may also influence treatment costs.

# group comparison:


In [32]:
# Summary statistics by readmission status

summary = PrData.groupby('Readmitted_30_Days')[
    ['Length_of_Stay', 'Treatment_Cost']
].mean()

print(summary.round(2))

                    Length_of_Stay  Treatment_Cost
Readmitted_30_Days                                
No                            4.52        84601.39
Yes                           5.18        89891.17


Interpretation

Patients who were readmitted within 30 days had an average hospital stay of 5.18 days, compared with 4.52 days among patients who were not readmitted. This represents a difference of approximately 0.66 days.

Similarly, patients who were readmitted had a higher average treatment cost of approximately KSh 89,891, compared with KSh 84,601 among patients who were not readmitted. This represents a difference of approximately KSh 5,290.

Overall finding

The descriptive analysis suggests that patients who experienced 30-day readmission tended to have longer hospital stays and higher treatment costs than those who were not readmitted. This may indicate that length of stay and treatment cost are associated with readmission outcomes. However, these differences alone do not establish statistical significance or causation.

To confirm this we need to test whether these differences are statistically significant using a t-test (or Mann–Whitney U test) for Length of Stay and Treatment Cost between the two readmission groups.

# Independent Samples t-test:
Does Length of Stay and Treatment Cost differ significantly between patients who were readmitted and those who were not?

use an independent samples t-test. If the data are not normally distributed, use the Mann–Whitney U test.

# Normality Test:

In [34]:
from scipy.stats import shapiro

# Length of Stay
los_no = PrData.loc[PrData['Readmitted_30_Days'] == 'No', 'Length_of_Stay'].dropna()
los_yes = PrData.loc[PrData['Readmitted_30_Days'] == 'Yes', 'Length_of_Stay'].dropna()

print("Length of Stay - Not Readmitted")
print(shapiro(los_no))

print("\nLength of Stay - Readmitted")
print(shapiro(los_yes))


# Treatment Cost
cost_no = PrData.loc[PrData['Readmitted_30_Days'] == 'No', 'Treatment_Cost'].dropna()
cost_yes = PrData.loc[PrData['Readmitted_30_Days'] == 'Yes', 'Treatment_Cost'].dropna()

print("\nTreatment Cost - Not Readmitted")
print(shapiro(cost_no))

print("\nTreatment Cost - Readmitted")
print(shapiro(cost_yes))

Length of Stay - Not Readmitted
ShapiroResult(statistic=np.float64(0.5942971443832276), pvalue=np.float64(5.432934312829266e-29))

Length of Stay - Readmitted
ShapiroResult(statistic=np.float64(0.6378676138456961), pvalue=np.float64(7.625916300174844e-16))

Treatment Cost - Not Readmitted
ShapiroResult(statistic=np.float64(0.9729513385567513), pvalue=np.float64(1.4238363403046036e-06))

Treatment Cost - Readmitted
ShapiroResult(statistic=np.float64(0.9510971075430901), pvalue=np.float64(0.00024404831979849348))


Not normally distributed , use mann-whitney-u

In [35]:
from scipy.stats import mannwhitneyu

# Length of Stay
u_los, p_los_mw = mannwhitneyu(
    los_no,
    los_yes,
    alternative='two-sided'
)

print("Length of Stay - Mann-Whitney U")
print("U-statistic:", u_los)
print("p-value:", p_los_mw)


# Treatment Cost
u_cost, p_cost_mw = mannwhitneyu(
    cost_no,
    cost_yes,
    alternative='two-sided'
)

print("\nTreatment Cost - Mann-Whitney U")
print("U-statistic:", u_cost)
print("p-value:", p_cost_mw)

Length of Stay - Mann-Whitney U
U-statistic: 21655.0
p-value: 0.25594090509508904

Treatment Cost - Mann-Whitney U
U-statistic: 21971.0
p-value: 0.367836301585886


A Mann–Whitney U test was conducted to assess whether Length of Stay and Treatment Cost differed significantly between patients who were readmitted within 30 days and those who were not. For Length of Stay, the test produced a U-statistic of 21,655 and a p-value of 0.256. Since the p-value was greater than 0.05, there was no statistically significant difference in Length of Stay between the two groups.

For Treatment Cost, the Mann–Whitney U test produced a U-statistic of 21,971 and a p-value of 0.368. Since the p-value was also greater than 0.05, there was no statistically significant difference in Treatment Cost between readmitted and non-readmitted patients.

Although readmitted patients had a higher average Length of Stay (5.18 days versus 4.52 days) and higher average Treatment Cost (KSh 89,891.17 versus KSh 84,601.39), these observed differences were not statistically significant at the 5% significance level. 

# Conclusion:
Therefore, the analysis does not provide sufficient statistical evidence that Length of Stay or Treatment Cost is associated with 30-day readmission in this dataset.
